In [34]:
# load data 
import numpy as np
import ROOT

OUTPUT_DIR = "/Users/youting/hgcal_energy_analysis/regression"
DATA_PATH = "/Users/youting/hgcal_energy_analysis/hgcal_dataset.npz"
d = np.load(DATA_PATH)

measuredE = d["measuredE"]   # (N, 34)
absorberE = d["absorberE"]
totalE    = d["totalE"]
voldEdx   = d["voldEdx"]
beam_E    = d["beam_energy"]  # GeV
run_num   = d["run_number"]

for k in d.files:
    print(f"{k:12s} {d[k].shape} {d[k].dtype}")

measuredE    (307424, 34) float64
absorberE    (307424, 34) float64
totalE       (307424, 34) float64
voldEdx      (307424, 34) float64
event_id     (307424,) uint32
beam_energy  (307424,) int64
run_number   (307424,) int64
source_file  (307424,) <U52


In [ ]:
# d["measuredE"] looks like (row,column), row = events number, column = section of materials

In [35]:
# linear regression
# train & test?

# From TB2025 Geometry
SILICON_IDX = [2, 4, 8, 10, 14, 16, 20, 22, 26, 31]
ABSORBER_IDX = [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 29, 32, 33]
EMPTY_IDX  = [1, 5, 7, 11, 13, 17, 19, 23, 25, 28, 30]


X = measuredE[:,SILICON_IDX] #input nparray[:, 要取的] (:表示row全取, SILICON_IDX表示column只取SILICON_IDX)
y = beam_E #goal

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y values: {np.unique(y)}")



X shape: (307424, 10)
y shape: (307424,)
y values: [ 20  50  80 100 150 200 300]


In [36]:
# split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)


#print(f"train(X, y): ({X_train.shape},{y_train.shape})")
#print(f"test(X, y): ({X_test.shape},{y_test.shape})")

#for E in np.unique(y):
#    n_tr = (y_train == E).sum()
#    n_te = (y_test == E).sum()

#    print(f"{E:.4f}GeV, train:{n_tr:6d}, test:{n_te:6d}, testing ratio:{(n_te)/(n_tr + n_te):.4f}")


In [4]:
# calculation
#X_train_std = np.std(X_train, axis = 0)
#X_test_std = np.std(X_test, axis = 0)
#print(X_train_std.shape)

(10,)


In [37]:
# differenciate beam energies
energies = np.unique(beam_E)
profile_mean = {}
profile_std = {}
test_mean = {}
test_std = {}

for E in energies:
    mask = (y_train == E)
    selected = X_train[mask] #Boolean mask
    profile_mean[E] = selected.mean(axis = 0)
    profile_std[E] = selected.std(axis = 0)

for E in energies:
    mask = (y_test == E)
    selected = X_test[mask]
    test_mean[E] = selected.mean(axis = 0)
    test_std[E] = selected.std(axis = 0)

In [38]:
# validation plot
def ValidationPlot(variable, variable_std, sample_type, beam_energy, color):
    x = np.array(SILICON_IDX, dtype = "float64")
    y = np.array(variable, dtype = "float64")
    ex = np.zeros_like(x)
    ey = np.array(variable_std, dtype = "float64")
    n = len(x)

    g_vali = ROOT.TGraphErrors(n, x, y, ex, ey)
    g_vali.SetName(f"g_vali_{sample_type}_{beam_energy}GeV")
    g_vali.SetMarkerStyle(24)
    g_vali.SetMarkerSize(1)
    g_vali.SetMarkerColor(color)
    g_vali.SetLineColor(color)
    g_vali.Draw("P SAME")
    return g_vali

In [39]:
# plotting (X_train validation)
colors = colors = [ROOT.kBlack, ROOT.kRed, ROOT.kBlue, ROOT.kGreen+2,
          ROOT.kMagenta, ROOT.kOrange+1, ROOT.kCyan+1]
# create Canvas
c = ROOT.TCanvas("c", "c", 900, 600)
frame = c.DrawFrame(0, 0, 33, 350)
frame.GetXaxis().SetTitle("section number")
frame.GetYaxis().SetTitle("measured energy(MeV)")

leg = ROOT.TLegend(0.15, 0.60, 0.35, 0.88)
graphs = []  # to avoid graph disappear

for E, col in zip(energies, colors):
    g_tr = ValidationPlot(profile_mean[E], profile_std[E], "measuredE(X_train)", E, col)
    graphs.append(g_tr)
    leg.AddEntry(g_tr, f"{E}GeV", "p")

leg.Draw()
c.Draw()
c.SaveAs(f"{OUTPUT_DIR}/X_train_validation.png")
c.SaveAs(f"{OUTPUT_DIR}/X_train_validation.root")

Info in <TCanvas::Print>: png file /Users/youting/hgcal_energy_analysis/regression/X_train_validation.png has been created
Info in <TCanvas::SaveAs>: ROOT file /Users/youting/hgcal_energy_analysis/regression/X_train_validation.root has been created


In [40]:
# plotting (X_test validation)
# create Canvas
c_te = ROOT.TCanvas("c_te", "c_te", 900, 600)
frame_te = c_te.DrawFrame(0, 0, 33, 350)
frame_te.GetXaxis().SetTitle("section number")
frame_te.GetYaxis().SetTitle("measured energy(MeV)")

leg_te = ROOT.TLegend(0.15, 0.60, 0.35, 0.88)
graphs_te = []  # to avoid graph disappear

for E, col in zip(energies, colors):
    g_te = ValidationPlot(test_mean[E], test_std[E], "measuredE(X_test)", E, col)
    graphs_te.append(g_te)
    leg_te.AddEntry(g_te, f"{E}GeV", "p")

leg_te.Draw()
c_te.Draw()
c_te.SaveAs(f"{OUTPUT_DIR}/X_test_validation.png")
c_te.SaveAs(f"{OUTPUT_DIR}/X_test_validation.root")

Info in <TCanvas::Print>: png file /Users/youting/hgcal_energy_analysis/regression/X_test_validation.png has been created
Info in <TCanvas::SaveAs>: ROOT file /Users/youting/hgcal_energy_analysis/regression/X_test_validation.root has been created


In [7]:
# fitting
# Using measuredE to fit E
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train) #default:using OLS

print("interception b:", model.intercept_)
for sec, w in zip(SILICON_IDX, model.coef_):
    print(f"section:{sec:2d}, weight:{w:.5f}")
print("train R square:", model.score(X_train, y_train)) 


interception b: 0.2833147920918009
section: 2, weight:0.16751
section: 4, weight:0.22642
section: 8, weight:0.21113
section:10, weight:0.21871
section:14, weight:0.21136
section:16, weight:0.20731
section:20, weight:0.20900
section:22, weight:0.22109
section:26, weight:0.31883
section:31, weight:0.65466
train R square: 0.9983717658180336


In [41]:
# predict
#y_pred = (X_test_k * model.coef_).sum(axis=1)+model.intercept_

y_pred = model.predict(X_test)


In [42]:
# Evaluation
print(f"{'E_true':>7s} {'E_pred_mean':>10s} {'E_pred_std':>9s} {'resolution':>11s} {'bias':>8s}")

mu_arr = np.zeros(len(energies))
sig_arr = np.zeros(len(energies))

E_arr = energies.astype("float64")

for i, E in enumerate(energies):
    mask = (y_test == E)
    mu_arr[i] = y_pred[mask].mean()
    sig_arr[i] = y_pred[mask].std()
    
    print(f"{E:7d} {mu_arr[i]:10.3f} {sig_arr[i]:9.3f} {sig_arr[i]/mu_arr[i]:11.4f} {(mu_arr[i]-E)/E:8.4f}")

rel_arr = sig_arr / mu_arr
bias_arr = ( mu_arr - E_arr )/ E_arr

 E_true E_pred_mean E_pred_std  resolution     bias
     20     20.225     1.379      0.0682   0.0113
     50     50.155     2.217      0.0442   0.0031
     80     80.147     2.899      0.0362   0.0018
    100    100.164     3.242      0.0324   0.0016
    150    147.263     4.039      0.0274  -0.0182
    200    200.055     4.750      0.0237   0.0003
    300    299.757     6.091      0.0203  -0.0008


In [43]:
# Linearity plot
def LinearityPlot(E_list, linearity_list, linearity_err_list, color, sample, leg_lin):
    x = np.array(E_list, dtype = "float64")
    y = np.array(linearity_list, dtype = "float64")
    ex = np.zeros_like(x)
    ey = np.array(linearity_err_list, dtype = "float64")
    n = len(x)
    
    g_lin = ROOT.TGraphErrors(n, x, y, ex, ey)
    g_lin.SetName(f"g_lin{sample}")
    g_lin.SetMarkerStyle(24)
    g_lin.SetMarkerSize(1)
    g_lin.SetMarkerColor(color)
    g_lin.SetLineColor(color)
    g_lin.Draw("P SAME")
    leg_lin.AddEntry(g_lin, f"{sample}", "lep")
    return g_lin


In [44]:
# plotting linearity
c_lin = ROOT.TCanvas("c_lin", "c_lin", 900, 600)
frame_lin = c_lin.DrawFrame(0, 0, 350, 450)
frame_lin.GetXaxis().SetTitle("beam E (GeV)")
frame_lin.GetYaxis().SetTitle("E_pred(GeV)")

leg_lin = ROOT.TLegend(0.15, 0.70, 0.30, 0.88)

g_lin = LinearityPlot(E_arr, mu_arr, sig_arr, ROOT.kBlue, "E_pred(linear)", leg_lin)

# fit
c_lin.cd()
ROOT.gStyle.SetOptFit(1111)
f_fit = ROOT.TF1("f_fit", "pol1", 0, 350) # p0 is intercept, p1 is slope
f_fit.SetLineColor(ROOT.kRed)
fit_result = g_lin.Fit(f_fit, "S")
leg_lin.AddEntry(f_fit, "linear fit", "l")

leg_lin.Draw()
c_lin.Draw()
c_lin.SaveAs(f"{OUTPUT_DIR}/E_pred_linearity.png")
c_lin.SaveAs(f"{OUTPUT_DIR}/E_pred_linearity.root")

****************************************
Minimizer is Linear / Migrad
Chi2                      =     0.392665
NDf                       =            5
p0                        =     0.324755   +/-   1.368       
p1                        =     0.994866   +/-   0.0158556   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_lin
Info in <TCanvas::Print>: png file /Users/youting/hgcal_energy_analysis/regression/E_pred_linearity.png has been created
Info in <TCanvas::SaveAs>: ROOT file /Users/youting/hgcal_energy_analysis/regression/E_pred_linearity.root has been created


In [ ]:
def ResolutionPlot():
    